In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyect0408") 

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta_tracks = f"abfss://{container}@{storageName}.dfs.core.windows.net/dataset.csv"


In [0]:
tracks_schema = StructType(fields=[
    StructField("Unnamed: 0", IntegerType(), True),
    StructField("track_id", StringType(), True),
    StructField("artists", StringType(), True),
    StructField("album_name", StringType(), True),
    StructField("track_name", StringType(), True),
    StructField("popularity", IntegerType(), True),
    StructField("duration_ms", IntegerType(), True),
    StructField("explicit", BooleanType(), True),
    StructField("danceability", DoubleType(), True),
    StructField("energy", DoubleType(), True),
    StructField("key", IntegerType(), True),
    StructField("loudness", DoubleType(), True),
    StructField("mode", IntegerType(), True),
    StructField("speechiness", DoubleType(), True),
    StructField("acousticness", DoubleType(), True),
    StructField("instrumentalness", DoubleType(), True),
    StructField("liveness", DoubleType(), True),
    StructField("valence", DoubleType(), True),
    StructField("tempo", DoubleType(), True),
    StructField("time_signature", IntegerType(), True),
    StructField("track_genre", StringType(), True)
])


In [0]:
df_tracks = spark.read\
    .option('header', True)\
    .schema(tracks_schema)\
    .csv(ruta_tracks)


In [0]:
tracks_selected_df = df_tracks.select(
    col("Unnamed: 0"), col("track_id"), col("artists"), col("album_name"), 
    col("track_name"), col("popularity"), col("duration_ms"), col("explicit"), 
    col("danceability"), col("energy"), col("key"), col("loudness"), 
    col("mode"), col("speechiness"), col("acousticness"), col("instrumentalness"), 
    col("liveness"), col("valence"), col("tempo"), col("time_signature"), col("track_genre")
)


In [0]:
tracks_final_df = tracks_selected_df.withColumn("ingestion_date", current_timestamp())

In [0]:

tracks_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.spotify_tracks")